# jev-judge Quickstart

Batch-judge items with the TypeSafe Jev API. This notebook needs no API key: it installs the package, shows the command line, and summarizes a small result file that ships next to it.

## 1) Install

Installs this checkout when the notebook runs from a clone of the repository (the CI case), otherwise the latest from GitHub. The install is non-editable on purpose: an editable install adds its `.pth` only at interpreter start, so a kernel that is already running would not see the package.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Install uv into this kernel's environment first: a bare `!uv` needs uv on PATH.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "uv"], check=True)
# The console scripts land next to the interpreter, which a plain kernel may not have on PATH.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

root = next(
    (p for p in (Path.cwd(), Path.cwd().parent) if (p / "src" / "btx_skill_jev_judge").is_dir()),
    None,
)
source = str(root) if root else "git+https://github.com/bitranox/btx-skill-jev-judge.git"
sample = (root / "notebooks" if root else Path.cwd()) / "sample_rows.jsonl"
source  # shown as the cell result: where the install comes from

In [ ]:
!{sys.executable} -m uv pip install --python {sys.executable} --upgrade {source}
assert _exit_code == 0

## 2) Help

Three commands: `run` judges items, `summarize` describes a result file, `check-key` tests that an API key is found.

In [ ]:
!jev-judge --help
assert _exit_code == 0

## 3) Is a key available?

`check-key` exits 0 when a key is found (an exported `TYPESAFE_API_KEY`, else a `.env` in the current directory or a parent, else `~/.credentials/typesafe.key`) and 1 when not. It never prints the key. To give the same answer on every machine, the next cell points the home directory at an empty temporary folder and drops the variable, and the check runs from that folder, so neither the keyfile nor a `.env` of yours can be found.

In [ ]:
import tempfile

empty_home = tempfile.mkdtemp()
os.environ["HOME"] = empty_home
os.environ["USERPROFILE"] = empty_home
os.environ.pop("TYPESAFE_API_KEY", None)

In [ ]:
!cd {empty_home} && jev-judge check-key --json
assert _exit_code == 1

## 4) Summarize a result file

`sample_rows.jsonl` holds 12 result rows for two yes/no questions. `is_bug` varies from item to item. `is_polite` answers 0.5 every time, so it tells the items apart not at all: `summarize` lists it under `flat` and exits 1, because a question like that is broken until shown otherwise.

In [ ]:
!jev-judge summarize --rows {sample} --json
assert _exit_code == 1